# A ridge path with an unpenalized intercept

An engineer predicts a building's daily energy use from 25 sensor readings, many of them
correlated, with only 40 days of training data. Least squares overfits, so you will fit
ridge regression with the intercept left out of the penalty, compute the weights for a
whole list of penalty strengths from one SVD, and let a validation set choose the strength.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# 25 correlated sensor readings around 20 units, energy use around 500.
rng = np.random.default_rng(11)
d = 25
mix = np.eye(d) + 0.4 * rng.normal(size=(d, d))
w_true = np.zeros(d)
w_true[:6] = [4.0, -3.0, 2.5, 2.0, -1.5, 1.0]
w_true[6:] = 0.3 * rng.normal(size=d - 6)


def sample_days(n):
    readings = 20 + rng.normal(size=(n, d)) @ mix
    energy = 500 + (readings - 20) @ w_true + rng.normal(scale=6.0, size=n)
    return readings, energy


X_train, y_train = sample_days(40)   # 40 days to fit on
X_val, y_val = sample_days(200)      # 200 later days to choose lambda on
lams = np.array([0, 0.1, 0.3, 1, 3, 10, 30, 100, 300, 1000, 3000.0])
print(X_train.shape, X_val.shape)

## Part 1 (coding task): Ridge with the intercept left alone

Write `ridge_fit(X, y, lam)` for a feature matrix `X` without a ones column. It returns
`(b, w)`, the intercept and the weights that minimize

$$
\lVert \mathbf{y} - b\,\mathbf{1} - X\mathbf{w} \rVert^2 + \lambda \lVert \mathbf{w} \rVert^2 ,
$$

with $b$ not penalized. Adding a ones column to $X$ and solving the ridge equations with it
charges the penalty on $b$ as well, which is wrong. Instead,
for any $\mathbf{w}$ the best $b$ is $\bar{y} - \bar{\mathbf{x}}^\top\mathbf{w}$, so center the
columns of $X$ and the target, solve ridge on the centered data without an intercept, and
recover $b$.

In [ ]:
def ridge_fit(X, y, lam):
    """Ridge regression with an unpenalized intercept. Returns (b, w)."""
    # TODO
    raise NotImplementedError

In [ ]:
b1, w1 = ridge_fit(X_train, y_train, 30.0)
b2, w2 = ridge_fit(X_train, y_train + 1000.0, 30.0)
assert np.allclose(w1, w2) and np.isclose(b2, b1 + 1000.0), "adding 1000 to every target should add 1000 to b and leave w alone: is the intercept being penalized?"

b, w = ridge_fit(np.array([[1.0], [2.0], [4.0], [5.0]]), np.array([3.0, 5.0, 4.0, 8.0]), 6.0)
assert np.isclose(w[0], 9 / 16) and np.isclose(b, 53 / 16), f"the four-point check expected b = 53/16 = 3.3125 and w = 9/16 = 0.5625, got b = {b:.4f}, w = {w[0]:.4f}"

b0, w0 = ridge_fit(X_train, y_train, 0.0)
design = np.hstack([np.ones((len(X_train), 1)), X_train])
ref = np.linalg.lstsq(design, y_train, rcond=None)[0]
assert np.allclose(np.r_[b0, w0], ref, atol=1e-6), "with lam = 0 the fit should be ordinary least squares with an intercept"
assert np.linalg.norm(w1) < np.linalg.norm(w0), "a positive lam should shrink the weights"
print("Part 1 passed")

## Part 2 (coding task): Every lambda from one SVD

Write `ridge_path(X, y, lams)`, which returns `(bs, W)`: an array of intercepts, one per
value in `lams`, and an array `W` of shape `(len(lams), d)` whose row $k$ is the weight
vector for `lams[k]`.

Factor the centered matrix once, $X_c = U \operatorname{diag}(s) V^\top$ with
`np.linalg.svd(Xc, full_matrices=False)`. Substituting it into the ridge solution gives

$$
\mathbf{w}(\lambda) = V \operatorname{diag}\!\Big(\frac{s_j}{s_j^2 + \lambda}\Big) U^\top \mathbf{y}_c ,
$$

so each $\lambda$ costs only a product with a small matrix. The formula also works when
$X_c$ has dependent columns, as long as $\lambda > 0$.

In [ ]:
def ridge_path(X, y, lams):
    """Intercepts and weights for every lam in lams, from one SVD of the centered X."""
    # TODO
    raise NotImplementedError

In [ ]:
bs, W = ridge_path(X_train, y_train, lams)
assert W.shape == (len(lams), d) and bs.shape == (len(lams),), f"expected shapes {(len(lams),)} and {(len(lams), d)}, got {np.shape(bs)} and {np.shape(W)}"
for k, lam in enumerate(lams):
    b_ref, w_ref = ridge_fit(X_train, y_train, lam)
    assert np.allclose(W[k], w_ref, atol=1e-6) and np.isclose(bs[k], b_ref), f"the weights for lam = {lam} do not match ridge_fit"

twin = np.column_stack([X_train[:, :3], X_train[:, 0]])      # column 3 copies column 0
_, W_twin = ridge_path(twin, y_train, [1.0])
assert np.all(np.isfinite(W_twin)) and np.isclose(W_twin[0, 0], W_twin[0, 3]), "with a copied column and lam > 0 the weight should split evenly between the two copies"
print("Part 2 passed")

## Part 3 (coding task): Let the validation set choose

Write `choose_lambda(X_tr, y_tr, X_va, y_va, lams)`. Fit the whole path on the training
rows, compute the mean squared error on the validation rows for every $\lambda$, intercept
included in the predictions, and return `(best_lam, val_mse)`: the $\lambda$ with the
smallest validation error and the array of all the validation errors.

In [ ]:
def choose_lambda(X_tr, y_tr, X_va, y_va, lams):
    # TODO
    raise NotImplementedError

In [ ]:
best, val_mse = choose_lambda(X_train, y_train, X_val, y_val, lams)
train_mse = [np.mean((y_train - b - X_train @ w) ** 2) for b, w in (ridge_fit(X_train, y_train, lam) for lam in lams)]
print(f"{'lam':>7} {'train MSE':>10} {'val MSE':>9}")
for lam, tr, va in zip(lams, train_mse, val_mse):
    print(f"{lam:>7g} {tr:>10.1f} {va:>9.1f}")
ref = [np.mean((y_val - b - X_val @ w) ** 2) for b, w in (ridge_fit(X_train, y_train, lam) for lam in lams)]
assert np.allclose(val_mse, ref), "the validation errors are wrong: predict with b + X_va @ w for each lam"
assert best == lams[np.argmin(ref)], "return the lam with the smallest validation error"
assert best != lams[0], "the training error always prefers the smallest lam: choose on the validation rows"
print(f"chosen lam = {best:g}: validation MSE {min(ref):.1f} against {ref[0]:.1f} for least squares")
print("Part 3 passed")

## Solution

### Part 1: Ridge with the intercept left alone

In [ ]:
def ridge_fit(X, y, lam):
    """Ridge regression with an unpenalized intercept. Returns (b, w)."""
    x_mean, y_mean = X.mean(axis=0), y.mean()
    Xc = X - x_mean
    gram = Xc.T @ Xc
    gram[np.diag_indices_from(gram)] += lam            # lam on every feature weight, none on b
    w = np.linalg.solve(gram, Xc.T @ (y - y_mean))
    return y_mean - x_mean @ w, w

**Why it works.** For fixed $\mathbf{w}$ the loss is a quadratic in $b$ minimized at
$b = \bar{y} - \bar{\mathbf{x}}^\top\mathbf{w}$. Putting that back in replaces $X$ and $\mathbf{y}$
by their centered versions, which leaves an ordinary ridge problem in $\mathbf{w}$ alone. The
shift check is the property that fails when the intercept is penalized: moving every target
by 1000 should move only $b$, but a penalty on $b$ resists the move and lets the weights
absorb part of it. Here the features sit near 20, so a ones column in the penalty can fake
part of the intercept through the weights, and the predictions go wrong in a way that is
easy to miss.

### Part 2: Every lambda from one SVD

In [ ]:
def ridge_path(X, y, lams):
    """Intercepts and weights for every lam in lams, from one SVD of the centered X."""
    x_mean, y_mean = X.mean(axis=0), y.mean()
    U, s, Vt = np.linalg.svd(X - x_mean, full_matrices=False)
    Uty = U.T @ (y - y_mean)
    lams = np.asarray(lams, dtype=float)
    factors = s / (s ** 2 + lams[:, None])          # one row of shrunken 1 / s per lam
    W = (factors * Uty) @ Vt
    return y_mean - W @ x_mean, W

**Why it works.** With $X_c = U \operatorname{diag}(s) V^\top$, $X_c^\top X_c + \lambda I = V \operatorname{diag}(s^2 + \lambda) V^\top$
when $X_c$ has full column rank, so the inverse is $V \operatorname{diag}(1/(s^2 + \lambda)) V^\top$,
and multiplying by $X_c^\top \mathbf{y}_c = V \operatorname{diag}(s)\,U^\top\mathbf{y}_c$ gives the
formula. Each direction $\mathbf{v}_j$ of the data is shrunk by $s_j^2 / (s_j^2 + \lambda)$
compared with least squares, so directions with little spread lose most of their weight. A
copied column gives one $s_j = 0$, and the factor $s_j / (s_j^2 + \lambda)$ is then 0 instead
of $1/0$. The copy and the original enter only through their sum, and the penalty is
smallest with the sum split evenly.

### Part 3: Let the validation set choose

In [ ]:
def choose_lambda(X_tr, y_tr, X_va, y_va, lams):
    bs, W = ridge_path(X_tr, y_tr, lams)
    pred = bs[:, None] + W @ X_va.T                  # one row of predictions per lam
    val_mse = np.mean((pred - y_va) ** 2, axis=1)
    return lams[np.argmin(val_mse)], val_mse

**Why it works.** The training error can only rise with $\lambda$, because $\lambda = 0$ minimizes it by
definition, so choosing on the training rows would always return the smallest value. The
validation rows played no part in the fit, so their error estimates how each $\lambda$ does
on new days: it falls from 92 at $\lambda = 0$ to about 48 at $\lambda = 30$ and rises again
as the penalty starts to erase the real effects. That minimum is itself picked as the best of
eleven noisy numbers, so a third split, untouched until the end, would be needed to report
how well the chosen model does.